# Evidencia ETL: integracion y calidad comercial

Este notebook ejecuta el mismo proceso reproducible de `main.py` y muestra la evidencia antes/despues, el modelo analitico y los cuatro indicadores solicitados.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
sys.path.insert(0, str(Path.cwd()))
from main import main
main()

## 1. Comparacion cuantificable de calidad

La tabla demuestra el efecto de duplicados, nulos y claves repetidas antes y despues del tratamiento.

In [ ]:
comparison = pd.read_csv('../06_calidad/quality_comparison.csv')
display(comparison)
display(pd.read_csv('../06_calidad/log_rechazos.csv', encoding='utf-8-sig'))

## 2. Cuatro indicadores para el dashboard

- Ingresos por ventas: suma de `venta_neta`.
- Unidades vendidas: suma de `cantidad`.
- Margen bruto: ingresos menos costo de los productos.
- Numero de transacciones: conteo distinto de `id_venta`.

In [ ]:
import os
from sqlalchemy import create_engine

target = os.environ.get('ETL_TARGET', '').strip().lower()
database_url = os.environ.get('DATABASE_URL', '').strip()
use_postgres = target == 'postgres' or (target != 'sqlite' and bool(database_url))
if use_postgres:
    if database_url.startswith('postgres://'):
        database_url = 'postgresql+psycopg2://' + database_url[len('postgres://'):]
    elif database_url.startswith('postgresql://'):
        database_url = 'postgresql+psycopg2://' + database_url[len('postgresql://'):]
    engine = create_engine(database_url)
    source = 'Neon/PostgreSQL'
else:
    engine = create_engine('sqlite:///../03_dw/dw.db')
    source = 'SQLite local'
print(f'Fuente de datos: {source}')

fact_table = 'fact_ventas' if use_postgres else 'FactVentas'
kpis = pd.read_sql_query(
    'SELECT SUM(venta_neta) ingresos, SUM(cantidad) unidades, '
    'SUM(margen) margen_bruto, COUNT(DISTINCT id_venta) transacciones '
    f'FROM {fact_table}',
    engine,
)
display(kpis)
display(pd.read_sql_query(f'SELECT * FROM {fact_table} LIMIT 5', engine))

if use_postgres:
    display(pd.read_sql_query('SELECT * FROM stg_ventas LIMIT 10', engine))
    display(pd.read_sql_query('SELECT * FROM clean_ventas LIMIT 10', engine))
else:
    display(pd.read_csv('../01_datos/staging/stg_ventas.csv', dtype=str).head(10))
    display(pd.read_csv('../01_datos/staging/clean_ventas.csv', dtype=str).head(10))

display(pd.read_sql_query('SELECT * FROM v_ventas_mes ORDER BY anio, mes', engine))
display(pd.read_sql_query('SELECT * FROM v_ventas_categoria ORDER BY venta_neta DESC', engine))

## 3. Interpretacion y proyeccion

Los rechazos se excluyen del hecho para impedir ventas huerfanas. La proyeccion debe usar el universo valido y monitorear la tasa de rechazo: una subida de rechazos reduce cobertura y puede sesgar las tendencias por cliente, producto o sucursal.